In [6]:
import yfinance as yf

# Le digo qué acción quiero — usamos Bancolombia
ticker = yf.Ticker("CIB")  # CIB es Bancolombia en bolsa de NY

# Le pido la información básica del instrumento
info = ticker.info

print("Nombre:", info["longName"])
print("Mercado:", info["exchange"])
print("Moneda:", info["currency"])
print("Precio actual:", info["currentPrice"])

Nombre: Grupo Cibest S.A.
Mercado: NYQ
Moneda: USD
Precio actual: 80.75


In [7]:
import yfinance as yf
import pandas as pd

ticker = yf.Ticker("CIB")

# Pido precios de los últimos 30 días
precios = ticker.history(period="30d")

print(precios.head())

                                Open       High        Low      Close  Volume  \
Date                                                                            
2026-04-30 00:00:00-04:00  67.860001  68.379997  66.589996  68.190002  348700   
2026-05-01 00:00:00-04:00  68.080002  68.160004  66.379997  67.040001  282900   
2026-05-04 00:00:00-04:00  67.120003  67.589996  64.699997  65.220001  243500   
2026-05-05 00:00:00-04:00  65.160004  66.820000  63.889999  65.820000  362300   
2026-05-06 00:00:00-04:00  66.230003  66.980003  64.980003  66.839996  288500   

                           Dividends  Stock Splits  
Date                                                
2026-04-30 00:00:00-04:00        0.0           0.0  
2026-05-01 00:00:00-04:00        0.0           0.0  
2026-05-04 00:00:00-04:00        0.0           0.0  
2026-05-05 00:00:00-04:00        0.0           0.0  
2026-05-06 00:00:00-04:00        0.0           0.0  


In [8]:
import subprocess
subprocess.run([
    r"C:\Users\busta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe",
    "-m", "pip", "install", "pyarrow"
])

CompletedProcess(args=['C:\\Users\\busta\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', 'pyarrow'], returncode=0)

In [9]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# --- EXTRACT ---
print("Extrayendo datos de Yahoo Finance...")
ticker = yf.Ticker("CIB")
precios = ticker.history(period="30d")

# --- LOAD → BRONZE ---
# Creamos la carpeta si no existe
os.makedirs("bronze", exist_ok=True)

# Nombre del archivo con la fecha de hoy
fecha_hoy = datetime.today().strftime("%Y_%m_%d")
ruta_bronze = f"bronze/precios_CIB_{fecha_hoy}.parquet"

# Guardamos tal como llegó — sin tocar nada
precios.to_parquet(ruta_bronze)

print(f"✅ Guardado en Bronze: {ruta_bronze}")
print(f"   {len(precios)} filas, {len(precios.columns)} columnas")

Extrayendo datos de Yahoo Finance...
✅ Guardado en Bronze: bronze/precios_CIB_2026_06_11.parquet
   30 filas, 7 columnas


In [10]:
# --- TRANSFORM → SILVER ---
os.makedirs("silver", exist_ok=True)

df_silver = precios.copy()

# Limpiar: quedarnos solo con columnas útiles
df_silver = df_silver[["Open", "High", "Low", "Close", "Volume"]]

# Renombrar a español para el negocio
df_silver.columns = ["apertura", "maximo", "minimo", "cierre", "volumen"]

# Agregar columna de ticker
df_silver["ticker"] = "CIB"

# Quitar filas con nulos
df_silver = df_silver.dropna()

# Guardar Silver
ruta_silver = f"silver/precios_CIB_{fecha_hoy}.parquet"
df_silver.to_parquet(ruta_silver)
print(f"✅ Silver listo: {ruta_silver}")

# --- TRANSFORM → GOLD ---
os.makedirs("gold", exist_ok=True)

# Resumen para negocio: estadísticas del período
df_gold = df_silver.agg({
    "cierre":  ["mean", "min", "max"],
    "volumen": ["mean", "sum"]
}).round(2)

# Guardar Gold
ruta_gold = f"gold/resumen_CIB_{fecha_hoy}.parquet"
df_gold.to_parquet(ruta_gold)
print(f"✅ Gold listo: {ruta_gold}")
print("\nResumen del período:")
print(df_gold)

✅ Silver listo: silver/precios_CIB_2026_06_11.parquet
✅ Gold listo: gold/resumen_CIB_2026_06_11.parquet

Resumen del período:
      cierre      volumen
mean   68.50    397591.73
min    63.16          NaN
max    80.75          NaN
sum      NaN  11927752.00


In [11]:
import pandas as pd

# Leer los archivos que acabas de crear
bronze = pd.read_parquet("bronze/precios_CIB_2026_06_09.parquet")
silver = pd.read_parquet("silver/precios_CIB_2026_06_09.parquet")
gold   = pd.read_parquet("gold/resumen_CIB_2026_06_09.parquet")

print("BRONZE:", bronze.shape)
print("SILVER:", silver.shape)
print("GOLD:\n", gold)

BRONZE: (30, 7)
SILVER: (30, 6)
GOLD:
       cierre      volumen
mean   67.81    393207.07
min    63.16          NaN
max    74.90          NaN
sum      NaN  11796212.00


In [12]:
print(f"✅ Guardado en Bronze: {ruta_bronze}")
print("Pipeline ejecutado por Luisa - práctica Git")

✅ Guardado en Bronze: bronze/precios_CIB_2026_06_11.parquet
Pipeline ejecutado por Luisa - práctica Git


In [13]:
import pandas as pd

data = {
    "ticker":   ["CIB", "CIB", "EC", "EC", "BVC", "BVC"],
    "fecha":    ["2026-06-08", "2026-06-09", "2026-06-08", "2026-06-09", "2026-06-08", "2026-06-09"],
    "cierre":   [67.50, 68.10, 24.30, 24.80, None, 5.10],
    "volumen":  [120000, 95000, 80000, 88000, 30000, 32000]
}

df = pd.DataFrame(data)
print(df)

  ticker       fecha  cierre  volumen
0    CIB  2026-06-08    67.5   120000
1    CIB  2026-06-09    68.1    95000
2     EC  2026-06-08    24.3    80000
3     EC  2026-06-09    24.8    88000
4    BVC  2026-06-08     NaN    30000
5    BVC  2026-06-09     5.1    32000


In [14]:
# Solo CIB
cib = df[df["ticker"] == "CIB"]

# Solo días donde el cierre fue mayor a 50
caros = df[df["cierre"] > 50]

# Múltiples condiciones: CIB Y fecha 2026-06-09
ambos = df[(df["ticker"] == "CIB") & (df["fecha"] == "2026-06-09")]

In [15]:
print("Solo CIB:\n", cib)
print("\nSolo días caros:\n", caros)    
print("\nCIB y fecha 2026-06-09:\n", ambos)    

Solo CIB:
   ticker       fecha  cierre  volumen
0    CIB  2026-06-08    67.5   120000
1    CIB  2026-06-09    68.1    95000

Solo días caros:
   ticker       fecha  cierre  volumen
0    CIB  2026-06-08    67.5   120000
1    CIB  2026-06-09    68.1    95000

CIB y fecha 2026-06-09:
   ticker       fecha  cierre  volumen
1    CIB  2026-06-09    68.1    95000


In [19]:
# Variación: si subió o bajó (necesitamos el día anterior)
df["valor_total"] = df["cierre"] * df["volumen"]
print(df[["ticker", "fecha", "valor_total"]])

  ticker       fecha  valor_total
0    CIB  2026-06-08    8100000.0
1    CIB  2026-06-09    6469500.0
2     EC  2026-06-08    1944000.0
3     EC  2026-06-09    2182400.0
4    BVC  2026-06-08          NaN
5    BVC  2026-06-09     163200.0


In [20]:
# Resumen por ticker
resumen = df.groupby("ticker").agg(
    precio_promedio = ("cierre", "mean"),
    precio_max      = ("cierre", "max"),
    volumen_total   = ("volumen", "sum")
).reset_index()

print(resumen)

  ticker  precio_promedio  precio_max  volumen_total
0    BVC             5.10         5.1          62000
1    CIB            67.80        68.1         215000
2     EC            24.55        24.8         168000


In [21]:
instrumentos = pd.DataFrame({
    "ticker": ["CIB", "EC", "BVC"],
    "nombre": ["Bancolombia", "Ecopetrol", "Bolsa de Valores Colombia"],
    "moneda": ["USD", "USD", "COP"]
})

# Combinar con el resumen
resumen_completo = resumen.merge(instrumentos, on="ticker")
print(resumen_completo)

  ticker  precio_promedio  precio_max  volumen_total  \
0    BVC             5.10         5.1          62000   
1    CIB            67.80        68.1         215000   
2     EC            24.55        24.8         168000   

                      nombre moneda  
0  Bolsa de Valores Colombia    COP  
1                Bancolombia    USD  
2                  Ecopetrol    USD  


In [ ]:
# Filtra solo las filas donde cierre no sea nulo
nonull = df[df["cierre"].notnull()]

In [25]:
#Crea un resumen por ticker con: precio promedio y volumen total
resumen = nonull.groupby("ticker").agg(
    precio_promedio = ("cierre", "mean"),
    volumen_total   = ("volumen", "sum")
).reset_index() 



In [26]:
#Une ese resumen con la tabla instrumentos para tener el nombre completo
resumen_completo = resumen.merge(instrumentos, on="ticker")
print(resumen_completo) 

  ticker  precio_promedio  volumen_total                     nombre moneda
0    BVC             5.10          32000  Bolsa de Valores Colombia    COP
1    CIB            67.80         215000                Bancolombia    USD
2     EC            24.55         168000                  Ecopetrol    USD


In [27]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# Lista de instrumentos a consultar
tickers = ["CIB", "EC", "BVN"]  # Bancolombia, Ecopetrol, Buenaventura

# Traer datos de cada uno y juntarlos
dfs = []
for t in tickers:
    precios = yf.Ticker(t).history(period="10d")
    precios["ticker"] = t  # agregamos columna para identificar el instrumento
    dfs.append(precios)

# Unir todo en un solo DataFrame
df_todos = pd.concat(dfs)
print(df_todos.shape)
print(df_todos[["ticker"]].value_counts())

(30, 8)
ticker
CIB       10
EC        10
BVN       10
Name: count, dtype: int64


In [28]:
os.makedirs("bronze", exist_ok=True)
fecha_hoy = datetime.today().strftime("%Y_%m_%d")
ruta_bronze = f"bronze/precios_multi_{fecha_hoy}.parquet"

df_todos.to_parquet(ruta_bronze)
print(f"✅ Bronze: {ruta_bronze} — {len(df_todos)} filas")

✅ Bronze: bronze/precios_multi_2026_06_11.parquet — 30 filas


In [29]:
os.makedirs("silver", exist_ok=True)

df = pd.read_parquet(ruta_bronze)

# Resetear índice (la fecha está como índice, la queremos como columna)
df = df.reset_index()

# Seleccionar y renombrar columnas
df_silver = df[["Date", "ticker", "Open", "High", "Low", "Close", "Volume"]].copy()
df_silver.columns = ["fecha", "ticker", "apertura", "maximo", "minimo", "cierre", "volumen"]

# Limpiar nulos
df_silver = df_silver.dropna()

ruta_silver = f"silver/precios_multi_{fecha_hoy}.parquet"
df_silver.to_parquet(ruta_silver)
print(f"✅ Silver: {ruta_silver} — {len(df_silver)} filas")
print(df_silver.head())

✅ Silver: silver/precios_multi_2026_06_11.parquet — 30 filas
                      fecha ticker   apertura     maximo     minimo  \
0 2026-05-29 00:00:00-04:00    CIB  68.910004  70.120003  68.419998   
1 2026-06-01 00:00:00-04:00    CIB  74.599998  77.889999  73.230003   
2 2026-06-02 00:00:00-04:00    CIB  73.769997  75.419998  72.480003   
3 2026-06-03 00:00:00-04:00    CIB  72.940002  73.339996  72.190002   
4 2026-06-04 00:00:00-04:00    CIB  72.150002  73.680000  71.809998   

      cierre  volumen  
0  68.589996   713100  
1  73.370003  1476700  
2  73.750000   593400  
3  72.250000   381600  
4  72.330002   210800  


In [30]:
os.makedirs("gold", exist_ok=True)

# Resumen por ticker
resumen = df_silver.groupby("ticker").agg(
    precio_promedio = ("cierre", "mean"),
    precio_max      = ("cierre", "max"),
    precio_min      = ("cierre", "min"),
    volumen_total   = ("volumen", "sum")
).reset_index().round(2)

# Tabla de instrumentos (en la realidad vendría de otra fuente)
instrumentos = pd.DataFrame({
    "ticker":  ["CIB", "EC", "BVN"],
    "nombre":  ["Bancolombia", "Ecopetrol", "Buenaventura"],
    "mercado": ["Colombia", "Colombia", "Peru"]
})

# Enriquecer el resumen
gold = resumen.merge(instrumentos, on="ticker")

ruta_gold = f"gold/resumen_multi_{fecha_hoy}.parquet"
gold.to_parquet(ruta_gold)
print(f"✅ Gold: {ruta_gold}")
print(gold)

✅ Gold: gold/resumen_multi_2026_06_11.parquet
  ticker  precio_promedio  precio_max  precio_min  volumen_total  \
0    BVN            32.99       36.89       30.23       14429034   
1    CIB            73.46       80.83       68.59        5637034   
2     EC            15.69       16.26       14.61       29970725   

         nombre   mercado  
0  Buenaventura      Peru  
1   Bancolombia  Colombia  
2     Ecopetrol  Colombia  
